# Critical Minerals Supply-Chain Risk Analytics: India's Ni-Co-Mn Exposure

SQLite is the primary tool here. Python and Pandas handle the concentration
arithmetic and the risk framework, and Matplotlib draws the dashboard. There is no
machine learning in this project.

One rule governs everything below: no number is invented. Every figure traces to a cell
in the source workbook or to arithmetic on workbook cells, and every gap is published as
a gap. That rule is why the Strategic Vulnerability Score at the end is deliberately left
uncomputed.

## 1. Setup

In [1]:
import os
import sqlite3

import pandas as pd

import load_data
import explore_data
import run_sql
import analysis

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

XLSX = "Critical_Minerals_NiCoMn_Dataset.xlsx"
DB = "critical_minerals.db"

## 2. Load the workbook into SQLite

Every sheet carries a title in row 1, a source note in row 2, a blank row 3 and the real
header in row 4, so every read uses `header=3`. README is skipped.

Five cleaning rules run at load time and each one exists because the raw sheet would
otherwise produce a wrong answer:

1. The CR3/HHI summary block at the bottom of `Global_Mining_2024` is not loaded as
   country rows. A row is kept only when the mineral is one of the three, the country
   value is genuine text and production is numeric. The block is loaded separately so the
   workbook's own figures can be compared against the recomputed ones.
2. `WORLD TOTAL`, `Other`, `INDIA TOTAL` and `India (Total)` are tagged in `row_type`
   rather than deleted, because the totals are needed as denominators and as
   reconciliation checks. Rankings filter on the tag.
3. Percentages are rescaled to 0 to 100 on load. The workbook stores them as fractions
   that are merely formatted as percent.
4. The manganese refining cell holds a sentence, not a number. It is kept verbatim and
   the numeric column is left NULL. Coercing it to 0 would assert a manganese refining
   concentration of zero, which is the opposite of what the source says.
5. Units are explicit in the column names, because the resource sheet is in thousand
   tonnes and the production sheet is in tonnes.

In [2]:
conn, frames = load_data.build_database(XLSX, DB)

SQLITE SCHEMA: critical_minerals.db

global_mining_2024  (21 rows)
    mineral                           TEXT
    country                           TEXT
    production_t                      REAL
    share_of_world_pct                REAL
    row_type                          TEXT

global_mining_concentration_workbook  (3 rows)
    mineral                           TEXT
    cr3_pct_workbook                  REAL
    hhi_workbook                      REAL
    concentration_read                TEXT
    basis                             TEXT

global_refining_shares  (5 rows)
    mineral                           TEXT
    country                           TEXT
    refining_share_raw                TEXT
    refining_share_pct                REAL
    is_numeric                        INTEGER

india_demand_projection  (2 rows)
    mineral                           TEXT
    demand_2025_t                     INTEGER
    demand_2026_t                     INTEGER
    demand_2027_t                

`global_mining_2024` holds 21 rows because 15 named country rows sit alongside 3 Other
buckets and 3 world totals. The resource and production tables each keep their national
total row, tagged and excluded from rankings.

In [3]:
pd.read_sql_query(
    "SELECT mineral, country, production_t, share_of_world_pct, row_type "
    "FROM global_mining_2024 ORDER BY mineral, production_t DESC",
    conn,
)

,mineral,country,production_t,share_of_world_pct,row_type
0,Cobalt,WORLD TOTAL,290000.0,NaN,world_total
1,Cobalt,DR Congo,220000.0,75.862069,country
2,Cobalt,Other,33300.0,11.482759,other_bucket
3,Cobalt,Indonesia,28000.0,9.655172,country
4,Cobalt,Russia,8700.0,3.000000,country
5,Manganese,WORLD TOTAL,20000000.0,NaN,world_total
6,Manganese,South Africa,7200000.0,36.000000,country
7,Manganese,Other,5400000.0,27.000000,other_bucket
8,Manganese,Gabon,4600000.0,23.000000,country
9,Manganese,Australia,2800000.0,14.000000,country


## 3. Exploratory data analysis

This runs after loading and before any metric, because the shape of the workbook is what
dictated every cleaning rule above. It corrects nothing. The correcting happens at load
time and this step shows the evidence that it was needed.

Three artefacts come out of it: a column level profile of all 8 tables, a matrix of which
minerals each table actually covers, and a list of data quality findings with live
evidence attached to each one.

In [4]:
profile, coverage, findings = explore_data.run(conn)

EDA 1  COLUMN PROFILE
51 columns across 8 tables. Columns carrying nulls:
                      table                   column  rows  nulls  null_pct
         global_mining_2024       share_of_world_pct    21      3      14.3
     global_refining_shares       refining_share_pct     5      1      20.0
india_mn_resources_by_state share_of_india_total_pct    12      1       8.3

EDA 2  MINERAL COVERAGE BY TABLE
mineral                      Nickel Cobalt Manganese
table                                               
global_mining_2024              yes    yes       yes
global_refining_shares          yes    yes        NO
india_demand_projection         yes    yes        NO
india_mn_production_by_state     NO     NO       yes
india_mn_resources_by_state      NO     NO       yes
india_nico_imports              yes    yes        NO
india_nico_resources            yes    yes        NO

Manganese is absent from 4 of the cross mineral tables, which is why no manganese import dependence and no man

The coverage matrix is the check that found the gaps. Manganese is absent from 4 of the
cross mineral tables, which is why no manganese import dependence, no manganese refining
share and no manganese demand projection exist anywhere downstream. That one matrix is
the reason the vulnerability score is blocked for manganese.

In [5]:
coverage.pivot(index="table", columns="mineral", values="covered")[
    ["Nickel", "Cobalt", "Manganese"]
].replace({True: "yes", False: "NO"})

mineral,Nickel,Cobalt,Manganese
table,,,
global_mining_2024,yes,yes,yes
global_refining_shares,yes,yes,NO
india_demand_projection,yes,yes,NO
india_mn_production_by_state,NO,NO,yes
india_mn_resources_by_state,NO,NO,yes
india_nico_imports,yes,yes,NO
india_nico_resources,yes,yes,NO


In [6]:
findings[["severity", "table", "finding", "impact"]]

,severity,table,finding,impact
0,high,global_mining_2024,summary block below the country rows carries a...,"a naive read creates countries named 0.774282,..."
1,high,global_mining_2024,aggregate rows are mixed in with country rows,ranking or summing without a filter double cou...
2,high,india_mn_resources_by_state and india_mn_produ...,a national total row sits inside the state lev...,ranking states without a filter puts the natio...
3,high,global_refining_shares,a text sentence occupies a numeric share column,coercing to 0 would assert a manganese refinin...
4,high,india_mn_resources_by_state vs india_mn_produc...,the two manganese tables use different units,an unconverted join is wrong by a factor of 1000
5,medium,all percentage columns,percentages are stored as fractions and merely...,charts and comparisons are off by a factor of 100
6,medium,india_mn_production_by_state,the production table does not cover every reso...,an inner join silently drops 3 states and a ra...
7,medium,india_nico_imports and india_nico_resources,"no customs volume series, and the two minerals...","a true dependence ratio cannot be built, and t..."
8,medium,global_refining_shares,refining coverage is partial by construction,"no refining HHI can be built, only a leader sh..."
9,low,india_nico_resources,mineral names carry a suffix that breaks joins,a direct join to the imports or demand table r...


## 4. SQL layer

All five queries live in one file, `sql/critical_minerals_analysis.sql`, separated by
`-- @query: <name>` marker comments. `run_sql.py` splits on those markers and executes
each section, so one file still produces one named CSV per query. The markers are
ordinary SQL comments, so the same file opens and runs section by section in DB Browser
for SQLite, DBeaver or the sqlite3 shell.

In [7]:
queries = run_sql.split_queries()
for name, sql in queries:
    print(name.ljust(32), len(sql.splitlines()), "lines")

01_country_production_rank       55 lines
02_import_dependence             97 lines
03_state_mn_resource_rank        64 lines
04_resource_vs_production        66 lines
05_cr3_hhi_concentration         103 lines


In [8]:
conn.close()
sql_results = run_sql.run_all(DB, verbose=False)
for name, df in sql_results.items():
    print(name.ljust(34), str(len(df)).rjust(3), "rows")
conn = sqlite3.connect(DB)

01_country_production_rank          15 rows
02_import_dependence                 3 rows
03_state_mn_resource_rank           11 rows
04_resource_vs_production           11 rows
05_cr3_hhi_concentration             3 rows


### Query 1: country production ranking

`RANK() OVER (PARTITION BY mineral ORDER BY production_t DESC)` for the rank and
`SUM() OVER (PARTITION BY mineral)` for the share. The Other bucket and the world total
are excluded from the ranking.

Two share columns are returned on purpose. `share_of_named_pct` is the `SUM() OVER`
basis, so the named countries sum to 100 percent. `share_of_world_pct` divides by the
workbook world total, which still contains Other, and that is the basis used for CR3 and
HHI later in the notebook. Reporting only the first would overstate every country's
position by deleting the unnamed producers from the world.

In [9]:
print(dict(run_sql.split_queries())["01_country_production_rank"])

-- ---------------------------------------------------------------------------
-- 01  Country production ranking per mineral, world mine production 2024
-- ---------------------------------------------------------------------------
-- Source table: global_mining_2024 (USGS Mineral Commodity Summaries 2025 and
-- USGS derived country estimates, loaded from Global_Mining_2024).
--
-- The Other bucket is excluded from the ranking because it is an aggregate of
-- many unnamed producers and is not a country. The WORLD TOTAL row is excluded
-- from the ranking for the same reason but is kept as a denominator.
--
-- Two share columns are reported on purpose and they answer different questions:
--   share_of_named_pct  = the SUM() OVER (PARTITION BY mineral) basis, so the
--                         named countries in this ranking sum to 100 percent.
--   share_of_world_pct  = the same production over the workbook WORLD TOTAL,
--                         which still includes the Other bucket. Th

In [10]:
sql_results["01_country_production_rank"]

,mineral,production_rank,country,production_t,share_of_named_pct,share_of_world_pct,named_countries_total_t,world_production_t,other_bucket_share_of_world_pct,cr3_member
0,Cobalt,1,DR Congo,220000.0,85.70,75.86,256700.0,290000.0,11.48,top_3
1,Cobalt,2,Indonesia,28000.0,10.91,9.66,256700.0,290000.0,11.48,top_3
2,Cobalt,3,Russia,8700.0,3.39,3.00,256700.0,290000.0,11.48,top_3
3,Manganese,1,South Africa,7200000.0,49.32,36.00,14600000.0,20000000.0,27.00,top_3
4,Manganese,2,Gabon,4600000.0,31.51,23.00,14600000.0,20000000.0,27.00,top_3
5,Manganese,3,Australia,2800000.0,19.18,14.00,14600000.0,20000000.0,27.00,top_3
6,Nickel,1,Indonesia,2200000.0,65.57,60.19,3355000.0,3655000.0,8.21,top_3
7,Nickel,2,Philippines,330000.0,9.84,9.03,3355000.0,3655000.0,8.21,top_3
8,Nickel,3,Russia,210000.0,6.26,5.75,3355000.0,3655000.0,8.21,top_3
9,Nickel,4,Canada,190000.0,5.66,5.20,3355000.0,3655000.0,8.21,rest


### Query 2: import dependence proxy

A `domestic_supply` CTE sets domestic mine production to 0 wherever
`active_mining_leases = 0`, then joins to the imports table. Where domestic production is
zero and imports are positive, dependence is 100 percent.

This is a structural proxy and the output says so on every row. The workbook has no
customs volume series for nickel, so a real ratio of imported volume over apparent
consumption cannot be built from it. Manganese is carried as an explicit gap row rather
than being left out, because India does mine manganese domestically and reusing the zero
supply assumption there would be wrong rather than merely unsupported.

In [11]:
print(dict(run_sql.split_queries())["02_import_dependence"])

-- ---------------------------------------------------------------------------
-- 02  Import dependence proxy for nickel and cobalt
-- ---------------------------------------------------------------------------
-- WHAT THIS IS: a structural proxy, not a customs volume calculation.
--
-- The workbook reports India holding 189 Mt of nickel ore resources and 45 Mt of
-- cobalt ore resources with ZERO active mining leases for both (India_NiCo_
-- Resources, National Mineral Inventory as on 01.04.2020). Zero active leases
-- means zero domestic mine supply, so every tonne consumed is imported and the
-- import dependence is 100 percent by construction.
--
-- It is NOT computed as imported volume divided by apparent consumption, because
-- the workbook carries no customs volume series for nickel and only an
-- approximate volume note for cobalt. The import VALUES below are the audited
-- figures and are labelled with currency and calendar year.
--
-- Manganese is carried as an explicit gap r

In [12]:
sql_results["02_import_dependence"][
    ["mineral", "import_year", "currency", "import_value_usd",
     "import_dependence_pct", "dependence_method"]
]

,mineral,import_year,currency,import_value_usd,import_dependence_pct,dependence_method
0,Cobalt,2023.0,USD,4.500000e+07,100.0,"PROXY: derived from zero active mining leases,..."
1,Manganese,NaN,NaN,NaN,NaN,DATA GAP: the workbook contains no manganese i...
2,Nickel,2024.0,USD,1.260000e+09,100.0,"PROXY: derived from zero active mining leases,..."


### Query 3: Indian states ranked by manganese resource share

Eleven states, with the INDIA TOTAL row excluded by `row_type`. The running
`cumulative_share_pct` is built in a second CTE, because SQLite will not accept a window
aggregate nested inside another window aggregate.

The `total_reconciliation` column compares the recomputed state sum against the workbook
national total. It returns `match` on all 11 rows.

In [13]:
print(dict(run_sql.split_queries())["03_state_mn_resource_rank"])

-- ---------------------------------------------------------------------------
-- 03  Indian states ranked by manganese total resource share
-- ---------------------------------------------------------------------------
-- Source table: india_mn_resources_by_state (IBM National Mineral Inventory as
-- on 01.04.2020, quantities in thousand tonnes).
--
-- The INDIA TOTAL row is tagged row_type = 'national_total' at load time and is
-- excluded here, so the 11 state rows are ranked on their own and their shares
-- sum to 100 percent. The audited national total is carried alongside as a check
-- column: recomputed state sum and the workbook INDIA TOTAL must agree.
-- ---------------------------------------------------------------------------

WITH national AS (
    SELECT total_resources_kt AS india_total_kt
    FROM   india_mn_resources_by_state
    WHERE  row_type = 'national_total'
),
states AS (
    SELECT state,
           reserves_kt,
           remaining_resources_kt,
           tot

In [14]:
sql_results["03_state_mn_resource_rank"][
    ["resource_rank", "state", "total_resources_mt", "share_of_india_pct",
     "cumulative_share_pct", "total_reconciliation"]
]

,resource_rank,state,total_resources_mt,share_of_india_pct,cumulative_share_pct,total_reconciliation
0,1,Odisha,171.528,34.06,34.06,match
1,2,Karnataka,123.972,24.62,58.67,match
2,3,Madhya Pradesh,60.057,11.92,70.60,match
3,4,Maharashtra,59.036,11.72,82.32,match
4,5,Goa,34.501,6.85,89.17,match
5,6,Andhra Pradesh,29.844,5.93,95.10,match
6,7,Jharkhand,14.749,2.93,98.03,match
7,8,Telangana,4.503,0.89,98.92,match
8,9,Gujarat,2.875,0.57,99.49,match
9,10,Rajasthan,2.359,0.47,99.96,match


### Query 4: resource base against FY2021-22 production

Resources are converted from thousand tonnes to tonnes before the ratio so both sides
share a unit. A LEFT JOIN keeps all 11 resource states and `production_status` labels
each one instead of dividing by zero: `no_production_row` for Goa, Gujarat and West
Bengal, `zero_production` for Jharkhand. The ratio is NULL for all four, never zero.

In [15]:
print(dict(run_sql.split_queries())["04_resource_vs_production"])

-- ---------------------------------------------------------------------------
-- 04  Manganese resource base against FY2021-22 production, by state
-- ---------------------------------------------------------------------------
-- Joins india_mn_resources_by_state (thousand tonnes, as on 01.04.2020) to
-- india_mn_production_by_state (tonnes, FY2021-22). Resources are converted to
-- tonnes before the ratio so both sides share one unit.
--
-- resource_t_per_t_production answers: for every tonne mined in FY2021-22, how
-- many tonnes sit in the resource base. A high number means a large endowment is
-- barely being worked.
--
-- No state is divided by zero and no missing state is dropped. A LEFT JOIN keeps
-- all 11 resource states and each untouched state is labelled:
--   no_production_row   Goa, Gujarat and West Bengal have resources but no row
--                       at all in the production table.
--   zero_production     Jharkhand appears in the production table with 0 tonnes
-- 

In [16]:
sql_results["04_resource_vs_production"][
    ["state", "total_resources_mt", "production_2021_22_t",
     "production_status", "resource_t_per_t_production"]
]

,state,total_resources_mt,production_2021_22_t,production_status,resource_t_per_t_production
0,Telangana,4.503,10435.0,producing,431.5
1,Odisha,171.528,512591.0,producing,334.6
2,Karnataka,123.972,380004.0,producing,326.2
3,Rajasthan,2.359,8008.0,producing,294.6
4,Andhra Pradesh,29.844,204002.0,producing,146.3
5,Maharashtra,59.036,731730.0,producing,80.7
6,Madhya Pradesh,60.057,849221.0,producing,70.7
7,Goa,34.501,NaN,no_production_row,NaN
8,Jharkhand,14.749,0.0,zero_production,NaN
9,Gujarat,2.875,NaN,no_production_row,NaN


### Query 5: CR3 and HHI in SQL

The same two concentration metrics the Python layer computes, built here with window
functions so the two can be cross checked. `RANK` is reported because it is the honest
competition rank and exposes ties, and nickel genuinely has one at position 6 where
Australia and New Caledonia are both on 110,000 t. `ROW_NUMBER` draws the top 3 cut so a
tie on the third position can never pull a fourth country into CR3.

The query also joins the workbook's own CR3/HHI block and reports the difference, which
is how the source error surfaces without anyone having to go looking for it.

In [17]:
print(dict(run_sql.split_queries())["05_cr3_hhi_concentration"])

-- ---------------------------------------------------------------------------
-- 05  CR3 and HHI per mineral, computed in SQL
-- ---------------------------------------------------------------------------
-- The same two concentration metrics that analysis.py computes, built here with
-- window functions so the result can be cross checked across both layers. The
-- pipeline asserts that the SQL and the Python agree to two decimal places.
--
--   CR3 = combined share of world mine production held by the top 3 NAMED
--         countries.
--   HHI = sum of squared percentage shares of the named countries, on the
--         0 to 10,000 scale.
--
-- Both use the workbook WORLD TOTAL as the denominator, which is the same basis
-- the workbook uses for its own Share of World column. The Other bucket sits in
-- that denominator but contributes no term of its own, because it aggregates
-- many unnamed producers and is not a country. Counting it as one producer
-- would square a sum instead of 

In [18]:
sql_results["05_cr3_hhi_concentration"][
    ["mineral", "named_countries", "cr3_named_pct", "top_1_country", "top_2_country",
     "top_3_country", "has_rank_tie", "hhi_named_lower_bound",
     "cr3_pct_workbook", "cr3_difference_pp", "hhi_workbook", "hhi_difference"]
]

,mineral,named_countries,cr3_named_pct,top_1_country,top_2_country,top_3_country,has_rank_tie,hhi_named_lower_bound,cr3_pct_workbook,cr3_difference_pp,hhi_workbook,hhi_difference
0,Cobalt,3,88.52,DR Congo,Indonesia,Russia,0,5857.0,97.00,-8.48,5989.0,-132.0
1,Nickel,9,74.97,Indonesia,Philippines,Russia,1,3798.0,77.43,-2.46,3865.0,-67.0
2,Manganese,3,73.00,South Africa,Gabon,Australia,0,2021.0,86.00,-13.00,2750.0,-729.0


## 5. Mining concentration: CR3 and HHI

CR3 is the combined share of world mine production held by the top three named countries.
HHI is the sum of squared percentage shares on the 0 to 10,000 scale. Both use the
workbook world total as the denominator.

The Other bucket is excluded from the CR3 numerator and from every HHI term, because it
aggregates many unnamed producers rather than being a country. Treating it as one
producer would square a sum instead of summing squares and overstate concentration.

The consequence is stated wherever the metric appears: **the HHI here is a lower bound**.
Splitting Other into its real constituent countries could only add positive terms. The
size of the omission travels with every value.

In [19]:
conc = analysis.concentration_metrics(conn)
conc[["mineral", "named_countries", "cr3_named_pct", "hhi_named_lower_bound",
      "other_share_of_world_pct", "cr3_top3", "hhi_interpretation"]]

,mineral,named_countries,cr3_named_pct,hhi_named_lower_bound,other_share_of_world_pct,cr3_top3,hhi_interpretation
0,Nickel,9,74.97,3798.0,8.21,"Indonesia, Philippines, Russia",highly concentrated (US DOJ/FTC threshold 2500)
1,Cobalt,3,88.52,5857.0,11.48,"DR Congo, Indonesia, Russia",highly concentrated (US DOJ/FTC threshold 2500)
2,Manganese,3,73.00,2021.0,27.00,"South Africa, Gabon, Australia",moderately concentrated


### SQL and Python agree

The Python above and SQL query 5 compute the same two metrics by different routes. The
pipeline asserts they agree, and they do to two decimal places on every mineral.

In [20]:
sql_conc = sql_results["05_cr3_hhi_concentration"]
cross = conc[["mineral", "cr3_named_pct", "hhi_named_lower_bound"]].merge(
    sql_conc[["mineral", "cr3_named_pct", "hhi_named_lower_bound"]],
    on="mineral", suffixes=("_python", "_sql"),
)
cross["cr3_agrees"] = (cross["cr3_named_pct_python"] - cross["cr3_named_pct_sql"]).abs() < 0.005
cross["hhi_agrees"] = (
    cross["hhi_named_lower_bound_python"] - cross["hhi_named_lower_bound_sql"]
).abs() < 0.5
cross

,mineral,cr3_named_pct_python,hhi_named_lower_bound_python,cr3_named_pct_sql,hhi_named_lower_bound_sql,cr3_agrees,hhi_agrees
0,Nickel,74.97,3798.0,74.97,3798.0,True,True
1,Cobalt,88.52,5857.0,88.52,5857.0,True,True
2,Manganese,73.00,2021.0,73.00,2021.0,True,True


### Against the workbook's own CR3/HHI block

The workbook appends its own concentration table, which sorts the Other bucket in with
the countries and therefore counts an aggregate as a single producer. The recomputed
figures are reported against it rather than quietly replacing it.

The cobalt case is the clearest: the workbook's 97 percent CR3 is built by ranking Other
second, above Indonesia. That is an artefact of the bucket, not a fact about the cobalt
market.

In [21]:
analysis.concentration_comparison(conn, conc)[
    ["mineral", "cr3_named_pct", "cr3_pct_workbook", "cr3_difference_pp",
     "hhi_named_lower_bound", "hhi_workbook", "hhi_difference"]
]

,mineral,cr3_named_pct,cr3_pct_workbook,cr3_difference_pp,hhi_named_lower_bound,hhi_workbook,hhi_difference
0,Nickel,74.97,77.428181,-2.46,3798.0,3865.321758,-67.0
1,Cobalt,88.52,97.000000,-8.48,5857.0,5989.129608,-132.0
2,Manganese,73.00,86.000000,-13.00,2021.0,2750.000000,-729.0


## 6. Demand CAGR 2025 to 2030

`(value_2030 / value_2025) ** (1/5) - 1`, five compounding periods between the endpoints.
Both results reproduce the workbook's own stored CAGR to within 1e-6, which the
`matches_workbook_cagr` column asserts.

There is no manganese demand projection in the workbook, so none is shown.

In [22]:
cagr = analysis.demand_cagr(conn)
cagr

,mineral,demand_2025_t,demand_2026_t,demand_2027_t,demand_2030_t,growth_multiple_2025_to_2030,cagr_2025_2030_pct,cagr_2025_2030_workbook_pct,matches_workbook_cagr,units
0,Cobalt,17,49,147,3878,228.12,196.23,196.231868,True,"tonnes per year, domestic requirement"
1,Nickel,2629,3057,6663,17492,6.65,46.09,46.086397,True,"tonnes per year, domestic requirement"


## 7. Strategic Vulnerability Score: designed, not computed

The intended composite is

```
score = 0.40 * import_dependence_pct
      + 0.35 * (mining_hhi / 100)
      + 0.25 * refining_leader_share_pct
```

with all three inputs normalised to a 0 to 100 scale before weighting, which is why the
HHI is divided by 100.

The gate is that all three inputs must exist and be complete for a mineral. No mineral
clears it, so **no score is computed for any mineral**. The reasons differ:

- Nickel and cobalt have a value in all three slots, but the import dependence is a
  structural proxy rather than a measured ratio, and the refining share comes from a USGS
  map listing only countries above 5 percent of world processing. A leader share drawn
  from a partial country list is not a concentration measurement.
- Manganese is missing two of the three inputs outright. No import table, no refining
  figure.

Publishing a weighted number on those inputs would present an assumption as a
measurement. The framework, the weights and the normalised inputs are all published so
the score can be computed the moment real customs data and a complete refining list are
added.

In [23]:
vuln = analysis.vulnerability_inputs(conn, conc)
vuln[["mineral",
      "weight_import_dependence", "input_import_dependence_pct", "quality_import_dependence",
      "weight_mining_hhi", "input_mining_hhi_scaled_0_100", "quality_mining_hhi",
      "weight_refining_leader", "input_refining_leader_pct", "quality_refining_leader",
      "score_status"]]

,mineral,weight_import_dependence,input_import_dependence_pct,quality_import_dependence,weight_mining_hhi,input_mining_hhi_scaled_0_100,quality_mining_hhi,weight_refining_leader,input_refining_leader_pct,quality_refining_leader,score_status
0,Nickel,0.4,100.0,proxy,0.35,37.98,lower_bound,0.25,56.0,partial,NOT COMPUTED
1,Cobalt,0.4,100.0,proxy,0.35,58.57,lower_bound,0.25,80.0,partial,NOT COMPUTED
2,Manganese,0.4,NaN,missing,0.35,20.21,lower_bound,0.25,NaN,missing,NOT COMPUTED


In [24]:
for _, r in vuln.iterrows():
    print(r["mineral"].ljust(11), r["score_blocked_reason"])

Nickel      INPUTS PRESENT BUT NOT COMPLETE: import_dependence, mining_hhi, refining_leader_share
Cobalt      INPUTS PRESENT BUT NOT COMPLETE: import_dependence, mining_hhi, refining_leader_share
Manganese   INPUTS ABSENT: import_dependence, refining_leader_share


## 8. Headline block

In [25]:
print(analysis.headline_block(conn, conc, cagr, vuln, sql_results))

HEADLINE NUMBERS  (every figure traced to the workbook)

[1] INDIA MANGANESE RESOURCE BASE  (IBM NMI as on 01.04.2020)
    States covered                 : 11
    India total resources          : 503.624 Mt
     1. Odisha            171.528 Mt    34.06 %
     2. Karnataka         123.972 Mt    24.62 %
     3. Madhya Pradesh     60.057 Mt    11.92 %
     4. Maharashtra        59.036 Mt    11.72 %
     5. Goa                34.501 Mt     6.85 %
    Odisha + Karnataka combined    : 58.67 %

[2] INDIA NICKEL AND COBALT RESOURCES  (NMI as on 01.04.2020)
    Cobalt          45 Mt ore   active mining leases: 0
    Nickel         189 Mt ore   active mining leases: 0

[3] IMPORT EXPOSURE  (value in US dollars, calendar year basis)
    Cobalt    US$ 45,000,000  (CY2023)   volume: ~1,000 tonnes
    Nickel    US$ 1,260,000,000  (CY2024)   volume: Not separately disclosed in source
    Manganese imports              : NOT IN WORKBOOK (gap)

[4] GLOBAL MINING CONCENTRATION 2024  (named countries onl

## 9. Dashboard

One self contained HTML file, no CDN and no external assets. Six panels: production share
by country per mineral with the top three highlighted, CR3 against HHI drawn as two charts
because they are two scales, the Indian state manganese distribution, import exposure
against the resource base that is not being mined, projected demand with a panel per
mineral, and the vulnerability inputs table marked SCORE NOT COMPUTED.

In [26]:
conn.close()
import build_dashboard
print("written:", build_dashboard.build())

written: /home/claude/project/critical-minerals/dashboard.html


## 10. Power BI layer

The same analysis is delivered a second time as a Power BI star schema and a five page
report spec. The export is not the analysis tables handed over as they stand. The
analysis database is shaped for SQL, where an aggregate row beside a detail row is
handled by a WHERE clause, and Power BI has no WHERE clause between a user and a chart.

So aggregates are split out by grain, year columns are unpivoted into rows, and the
dimensions are separated and keyed. `ref_concentration` carries the CR3 and HHI values
that both layers above agree on, which turns it into a test harness for the DAX.

In [27]:
import build_powerbi_export
import build_powerbi_mockup

tables = build_powerbi_export.build()
print()
print("mockup:", build_powerbi_mockup.build())

POWER BI STAR SCHEMA EXPORT
  dim   dim_mineral                3 rows    5 cols
  dim   dim_country               13 rows    3 cols
  dim   dim_state                 11 rows    3 cols
  dim   dim_year                   6 rows    5 cols
  fact  fact_world_mining         15 rows    8 cols
  ref   ref_world_totals           3 rows    7 cols
  fact  fact_mn_resources         11 rows    9 cols
  fact  fact_mn_production        24 rows    6 cols
  ref   ref_india_totals           2 rows    5 cols
  fact  fact_nico_resources        2 rows    9 cols
  fact  fact_imports               3 rows   12 cols
  fact  fact_demand                8 rows    6 cols
  ref   ref_demand_cagr            2 rows    6 cols
  fact  fact_refining              5 rows    8 cols
  ref   ref_concentration          3 rows   12 cols
  ref   ref_vulnerability          3 rows   15 cols
  ref   ref_data_quality          10 rows    8 cols

  CSVs      -> powerbi/data/
  Workbook  -> powerbi/Critical_Minerals_PowerBI.xlsx

moc

In [28]:
tables["fact_world_mining"].head(8)

,mineral_key,country_key,mineral,country,production_t,share_of_world_pct,production_rank,is_top_3
0,1,8,Nickel,Indonesia,2200000.0,60.1915,1,True
1,1,10,Nickel,Philippines,330000.0,9.0287,2,True
2,1,11,Nickel,Russia,210000.0,5.7456,3,True
3,1,3,Nickel,Canada,190000.0,5.1984,4,False
4,1,4,Nickel,China,120000.0,3.2832,5,False
5,1,1,Nickel,Australia,110000.0,3.0096,6,False
6,1,9,Nickel,New Caledonia,110000.0,3.0096,6,False
7,1,2,Nickel,Brazil,77000.0,2.1067,8,False


### The Power BI template

A .pbix is a compiled binary whose model part only Power BI itself can write, so one
cannot honestly be produced outside Power BI Desktop. A .pbit is the open equivalent: the
same container, with the model as TMSL JSON and the report as layout JSON. Power BI
Desktop opens it, builds the model, runs the queries and gives a live report to save as
.pbix.

The rows are embedded in the M as literals so the template opens anywhere with no file
path and no broken refresh. Since Power BI cannot be run here, `build_pbit.py` checks
what it can mechanically: the container and encodings, that every DAX, relationship,
visual field and filter reference resolves against the model, and that the embedded rows
match the export exactly.

In [29]:
import build_pbit

build_pbit.build()

POWER BI TEMPLATE
  tables         18
  relationships  14
  measures       32
  embedded rows  124
  report pages   5

  semantic check passed: every DAX, relationship, visual field and
  filter reference resolves against the model
  row check passed: the embedded tables match the export exactly

  Critical_Minerals_Report.pbit     26 KB   checks passed

  Critical_Minerals_Model.pbit      14 KB   checks passed


('/home/claude/project/critical-minerals/powerbi/Critical_Minerals_Report.pbit',
 '/home/claude/project/critical-minerals/powerbi/Critical_Minerals_Model.pbit')

## 11. Verification

Every headline figure is checked against the expected values for this project, including
the Power BI export.

In [30]:
import verify_expectations
verify_expectations.main()

MANGANESE STATE RESOURCES
PASS  Mn states counted                             got             11   expected             11
PASS  India Mn total, Mt                            got        503.624   expected        503.624
PASS  Odisha share pct                              got          34.06   expected          34.06
PASS  Odisha is rank 1                              got         Odisha   expected         Odisha
PASS  Karnataka share pct                           got          24.62   expected          24.62
PASS  Karnataka is rank 2                           got      Karnataka   expected      Karnataka
PASS  Odisha + Karnataka pct                        got          58.67   expected          58.68
      NOTE: exact value is 58.6747 pct. 58.68 is the sum of the two rounded shares (34.06 + 24.62). Rounding the exact sum gives 58.67. Both describe the same quantity and both round to 59 pct at resume precision.
PASS  Rank 3 state                                  got Madhya Pradesh   expected

PASS  model tables                                  got             18   expected             18
PASS  model relationships                           got             14   expected             14
PASS  model measures                                got             32   expected             32
PASS  report pages                                  got              5   expected              5
PASS  every model and report reference resolves     got    no problems   expected    no problems
PASS  embedded rows match the export                got    no problems   expected    no problems

RESULT: 111 of 111 checks passed


0